# 03 — Pandas Analysis (TMDB Movie Dataset)

Works through all 10 Pandas questions from the project brief, using the
cleaned CSVs from `01_data_cleaning.ipynb`.


## Setup — load the cleaned tables

In [1]:
import pandas as pd
import numpy as np

CLEAN_DIR = "data/cleaned"

movies = pd.read_csv(f"{CLEAN_DIR}/movies.csv")
genres = pd.read_csv(f"{CLEAN_DIR}/genres.csv")
movie_genres = pd.read_csv(f"{CLEAN_DIR}/movie_genres.csv")
cast = pd.read_csv(f"{CLEAN_DIR}/cast.csv")
crew = pd.read_csv(f"{CLEAN_DIR}/crew.csv")
movie_keywords = pd.read_csv(f"{CLEAN_DIR}/movie_keywords.csv")

# release_date gets saved as plain text in a CSV, so convert it back to a real date
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

print(f"movies: {len(movies)} | genres: {len(genres)} | movie_genres: {len(movie_genres)} | "
      f"cast: {len(cast)} | crew: {len(crew)} | movie_keywords: {len(movie_keywords)}")

movies: 2499 | genres: 19 | movie_genres: 6878 | cast: 24902 | crew: 7289 | movie_keywords: 41681


## Q1 — Total unique counts across all tables

List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [2]:
print("Unique movies:      ", movies["movie_id"].nunique())
print("Unique genres:      ", genres["genre_id"].nunique())
print("Unique cast members:", cast["person_id"].nunique())
print("Unique crew members:", crew["person_id"].nunique())

Unique movies:       2499
Unique genres:       19
Unique cast members: 11300
Unique crew members: 2848


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q2 — Movies with missing or zero budget/revenue

List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

In [3]:
missing_mask = (
    (movies["budget"] == 0) | (movies["budget"].isna()) |
    (movies["revenue"] == 0) | (movies["revenue"].isna())
)
missing_budget_revenue = movies[missing_mask]

print(f"{len(missing_budget_revenue)} of {len(movies)} movies "
      f"({len(missing_budget_revenue) / len(movies):.1%}) have missing or zero budget/revenue.")
missing_budget_revenue[["title", "budget", "revenue"]].head(10)

133 of 2499 movies (5.3%) have missing or zero budget/revenue.


,title,budget,revenue
324,Zack Snyder's Justice League,70000000,0
340,Bird Box,19800000,0
403,365 Days,0,9458590
468,To All the Boys I've Loved Before,0,0
554,Prey,65000000,0
598,The Kissing Booth,0,0
717,Enola Holmes,21000000,0
723,Extraction,65000000,0
754,"Love, Rosie",0,4439431
842,Bright,90000000,0


**Decision to make:** these are almost certainly *unknown* values (TMDB uses `0` as a placeholder), not real zeros. For any question below that involves budget/revenue math (profit, ROI), we exclude these rows rather than treating `0` as a real financial figure — otherwise "profit" for an unknown-budget movie would be wildly overstated.

**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q3 — Profit and ROI per movie

List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [4]:
valid_financials = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)].copy()
valid_financials["profit"] = valid_financials["revenue"] - valid_financials["budget"]
valid_financials["roi"] = valid_financials["profit"] / valid_financials["budget"]

print(f"{len(valid_financials)} movies have usable budget/revenue figures.")
valid_financials[["title", "budget", "revenue", "profit", "roi"]].head(10)

2366 movies have usable budget/revenue figures.


,title,budget,revenue,profit,roi
0,Interstellar,165000000,746606706,581606706,3.524889
1,Inception,160000000,839030630,679030630,4.243941
2,The Avengers,220000000,1518815515,1298815515,5.903707
3,The Dark Knight,185000000,1004558444,819558444,4.430046
4,Avatar,237000000,2923706026,2686706026,11.336312
5,Deadpool,58000000,782837347,724837347,12.497196
6,Fight Club,63000000,100853753,37853753,0.600853
7,Avengers: Infinity War,300000000,2052415039,1752415039,5.841383
8,The Shawshank Redemption,25000000,28341469,3341469,0.133659
9,Pulp Fiction,8000000,213928762,205928762,25.741095


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q4 — Top 15 movies by ROI

List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.

In [5]:
top_roi = (
    valid_financials[valid_financials["budget"] >= 1_000_000]
    .sort_values("roi", ascending=False)
    .head(15)
)
top_roi[["title", "budget", "revenue", "roi"]]

,title,budget,revenue,roi
1767,Dragon Ball Super: Broly,1000000,125002821,124.002821
535,Snow White and the Seven Dwarfs,1488423,184925486,123.242561
1791,The Rocky Horror Picture Show,1400000,171181400,121.272429
467,Rocky,1000000,117253345,116.253345
1227,Gone with the Wind,4000000,402352579,99.588145
716,The Jungle Book,4000000,378000000,93.500000
635,Cinderella,2900000,263600000,89.896552
372,Saw,1200000,104045735,85.704779
719,One Hundred and One Dalmatians,3600000,303000000,83.166667
263,E.T. the Extra-Terrestrial,10500000,797307407,74.934039


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q5 — Movies released per year

List the number of movies released per year, and identify which year had the highest movie output in the dataset.

In [6]:
movies["release_year"] = movies["release_date"].dt.year

yearly_counts = (
    movies.groupby("release_year")
    .size()
    .reset_index(name="movie_count")
    .sort_values("release_year")
)
yearly_counts

,release_year,movie_count
0,1921,1
1,1922,1
2,1927,1
3,1931,2
4,1936,1
...,...,...
84,2022,65
85,2023,62
86,2024,43
87,2025,26


In [7]:
busiest_year = yearly_counts.loc[yearly_counts["movie_count"].idxmax()]
print(f"Busiest year: {int(busiest_year['release_year'])} with {busiest_year['movie_count']} movies.")

Busiest year: 2016 with 121 movies.


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q6 — Runtime outliers

List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

We use the **IQR (interquartile range) method**: anything more than 1.5×IQR
below Q1 or above Q3 is flagged as an outlier. This is a standard,
beginner-friendly way to define an outlier without assuming a particular
distribution shape.

In [8]:
Q1 = movies["runtime"].quantile(0.25)
Q3 = movies["runtime"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

runtime_outliers = movies[(movies["runtime"] < lower_bound) | (movies["runtime"] > upper_bound)]

print(f"Normal runtime range: {lower_bound:.0f}–{upper_bound:.0f} minutes")
print(f"{len(runtime_outliers)} outlier movies found.")
runtime_outliers[["title", "runtime"]].sort_values("runtime")

Normal runtime range: 58–166 minutes
60 outlier movies found.


,title,runtime
2307,The Guardians of the Galaxy Holiday Special,45
2447,The Garden of Words,46
514,Dune: Part Two,167
620,Gangs of New York,168
110,Saving Private Ryan,169
397,It Chapter Two,169
0,Interstellar,169
146,Pirates of the Caribbean: At World's End,169
76,The Hobbit: An Unexpected Journey,169
492,Heat,170


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q7 — Average popularity by primary genre

List each movie's primary genre (the first genre listed) and use it to
compare average popularity across primary genres.

"Primary genre" = the lowest `genre_id` attached to that movie, used here
as a simple, consistent tie-breaking rule for movies with multiple genres.

In [9]:
movie_genres_named = movie_genres.merge(genres, on="genre_id")

primary_genre = (
    movie_genres_named.sort_values(["movie_id", "genre_id"])
    .groupby("movie_id")
    .first()
    .reset_index()[["movie_id", "genre_name"]]
    .rename(columns={"genre_name": "primary_genre"})
)

movies_with_primary = movies.merge(primary_genre, on="movie_id", how="left")

avg_popularity_by_genre = (
    movies_with_primary.groupby("primary_genre")["popularity"]
    .mean()
    .sort_values(ascending=False)
)
avg_popularity_by_genre

primary_genre
Thriller           18.854481
Adventure          17.372862
Animation          15.234531
Horror             14.600246
Crime              14.592000
Western            13.714286
Action             13.284634
Science Fiction    12.597567
Fantasy            12.424829
Drama              11.151012
History            10.754000
Comedy             10.148475
Documentary         7.726200
War                 3.218400
Name: popularity, dtype: float64

**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q8 — Top 10 prolific actors and their average rating

List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.

In [10]:
cast_with_ratings = cast.merge(movies[["movie_id", "vote_average"]], on="movie_id")

actor_stats = (
    cast_with_ratings.groupby("actor_name")
    .agg(movie_count=("movie_id", "nunique"), avg_rating=("vote_average", "mean"))
    .sort_values("movie_count", ascending=False)
    .head(10)
)
actor_stats

,movie_count,avg_rating
actor_name,,
Samuel L. Jackson,39,6.938692
Robert De Niro,38,7.228316
Brad Pitt,38,7.257289
Johnny Depp,37,6.856973
Tom Hanks,33,7.289758
Willem Dafoe,32,7.114781
Mark Wahlberg,32,6.630063
Scarlett Johansson,32,7.056344
Tom Cruise,31,7.040613


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q9 — Duplicate movie titles

List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.

In [11]:
duplicate_titles = movies[movies.duplicated("title", keep=False)].sort_values("title")
duplicate_titles[["title", "release_year", "movie_id"]]

,title,release_year,movie_id
878,A Nightmare on Elm Street,1984,377
1928,A Nightmare on Elm Street,2010,23437
339,Aladdin,2019,420817
261,Aladdin,1992,812
166,Alice in Wonderland,2010,12155
...,...,...,...
370,The Mummy,1999,564
532,The Thing,1982,1091
1737,The Thing,2011,60935
860,Total Recall,2012,64635


**Note:** not every match here is a data error — some are legitimate remakes (e.g., a 1970s film and its 2020s remake sharing the same title). Check the release years before assuming these are duplicates to clean up.

**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_

## Q10 — Movies where popularity and vote count disagree

List movies whose popularity score and vote count don't follow the general
pattern between the two — i.e., unusually high popularity with a low vote
count, or vice versa.

**Approach:** rank every movie by popularity and separately by vote count.
Normally a movie's two ranks should be fairly close (popular movies tend to
have lots of votes). We flag the movies with the **biggest gap** between
their two ranks as the ones breaking that pattern.

In [12]:
movies["popularity_rank"] = movies["popularity"].rank(ascending=False)
movies["vote_count_rank"] = movies["vote_count"].rank(ascending=False)
movies["rank_gap"] = (movies["popularity_rank"] - movies["vote_count_rank"]).abs()

mismatched = movies.sort_values("rank_gap", ascending=False).head(15)
mismatched[["title", "popularity", "popularity_rank", "vote_count", "vote_count_rank", "rank_gap"]]

,title,popularity,popularity_rank,vote_count,vote_count_rank,rank_gap
2452,Mortal Kombat II,99.5231,10.0,2260,2453.0,2443.0
2493,Lilo & Stitch,33.2161,98.0,2228,2494.0,2396.0
2388,Disclosure Day,394.3346,2.0,2328,2388.0,2386.0
2295,Lee Cronin's The Mummy,79.4601,15.0,2430,2296.0,2281.0
2441,Lolita,27.5391,161.0,2271,2441.5,2280.5
205,X-Men: First Class,1.2614,2486.0,13680,206.0,2280.0
2248,Backrooms,258.4743,4.0,2483,2250.0,2246.0
244,Home Alone,1.8114,2480.0,12658,245.0,2235.0
2457,Smile 2,23.7112,228.0,2256,2459.0,2231.0
254,X-Men,1.2978,2485.0,12340,255.0,2230.0


**Insight:** _(Write 2–4 sentences here — what does this result tell you, why does it matter, and what would you recommend based on it?)_